Notebook ini **hanya melakukan evaluation/inference**, tanpa training ulang.

Tujuan:
- menggunakan checkpoint **FrameMean** yang sudah dilatih;
- menguji sensitivitas terhadap jumlah **distinct temporal observations**;
- held-out domain dievaluasi terpisah:
  1. RWF-2000,
  2. UCF-Crime Fighting subset,
  3. Bus Violence.

Konfigurasi frame-diversity mengikuti notebook V3:

```text
T cache tetap = 16
K ∈ {1, 2, 4, 8, 16}
K=1/2/4/8 → 16 posisi dibagi menjadi K temporal bins
             → pilih 1 posisi acak dari setiap bin
             → 10 repeats
K=16       → gunakan seluruh 16 posisi
```

Pemilihan frame menggunakan `SAMPLING_BASE_SEED=20260810` dan **tidak bergantung
pada checkpoint seed**, sehingga kondisi sampling yang sama dipakai pada seed
0, 42, dan 123.

> Penting: `K=1` pada eksperimen ini **bukan** model SingleFrame yang dilatih
> terpisah. Ini adalah inference-time evidence-sensitivity test pada checkpoint
> FrameMean yang sama.

Untuk FrameMean, merata-ratakan K distinct features secara matematis ekuivalen
dengan menduplikasi setiap distinct feature secara sama hingga T=16 lalu
mengambil mean, karena K selalu membagi 16.

## Mengganti seed

Setelah path dataset benar, **cukup ubah satu baris**:

```python
RUN_SEED = 0
```

menjadi `42` atau `123`.

In [ ]:
# Jalankan sekali di awal sesi Google Colab.
!pip -q install timm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import gc
import json
import random
import platform
import warnings
import zlib
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
from timm import create_model
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
)

warnings.filterwarnings("ignore", category=UserWarning)

print("Python :", platform.python_version())
print("PyTorch:", torch.__version__)
print("timm   :", timm.__version__)
print("CUDA   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))

Python : 3.13.15
PyTorch: 2.11.0+cu128
timm   : 1.0.28
CUDA   : True
GPU    : NVIDIA A100-SXM4-40GB


## 1. Konfigurasi

### Satu-satunya variabel yang diubah antar-seed

```python
RUN_SEED = 0
```

Valid:

```python
0
42
123
```

Jangan ubah antar-seed:
- `K_VALUES = [1,2,4,8,16]`
- `NUM_RANDOM_REPEATS = 10`
- `SAMPLING_BASE_SEED = 20260810`
- shared 16-frame cache
- preprocessing
- checkpoint architecture
- held-out sample order

Dengan demikian perbedaan antar-seed berasal dari checkpoint training, bukan
dari perubahan frame selection.

In [ ]:
# ============================================================
# SATU-SATUNYA VARIABEL YANG DIUBAH ANTAR-SEED
# ============================================================
RUN_SEED = 123

VALID_SEEDS = (0, 42, 123)

if RUN_SEED not in VALID_SEEDS:
    raise ValueError(
        f"RUN_SEED harus salah satu dari {VALID_SEEDS}, mendapat {RUN_SEED}."
    )

# ============================================================
# MODEL / CHECKPOINT
# ============================================================
MODEL_ROOT = Path(
    "/content/drive/MyDrive/Models/"
)

VARIANT = "frame_mean"

CHECKPOINT_ROOT_CANDIDATES = [
    MODEL_ROOT / "Model_42Seeds",
    MODEL_ROOT / "Model_0Seeds",
    MODEL_ROOT / "Model_123Seeds_v1",

]

CHECKPOINT_PATH_OVERRIDE = None


def resolve_checkpoint(seed: int) -> Path:
    if CHECKPOINT_PATH_OVERRIDE is not None:
        path = Path(CHECKPOINT_PATH_OVERRIDE)

        if not path.exists():
            raise FileNotFoundError(
                f"CHECKPOINT_PATH_OVERRIDE tidak ditemukan: {path}"
            )

        return path

    direct_matches = []

    for root in CHECKPOINT_ROOT_CANDIDATES:
        candidate = (
            root
            / VARIANT
            / f"seed_{seed}"
            / "best.pt"
        )

        if candidate.exists():
            direct_matches.append(candidate)

    direct_matches = list(
        dict.fromkeys(direct_matches)
    )

    if len(direct_matches) == 1:
        return direct_matches[0]

    if len(direct_matches) > 1:
        raise RuntimeError(
            "Ditemukan lebih dari satu checkpoint FrameMean yang cocok. "
            "Set CHECKPOINT_PATH_OVERRIDE secara eksplisit:\n"
            + "\n".join(str(p) for p in direct_matches)
        )

    recursive_matches = sorted(
        MODEL_ROOT.glob(
            f"**/{VARIANT}/seed_{seed}/best.pt"
        )
    )

    if len(recursive_matches) == 1:
        return recursive_matches[0]

    if len(recursive_matches) == 0:
        raise FileNotFoundError(
            f"best.pt FrameMean seed {seed} tidak ditemukan di {MODEL_ROOT}.\n"
            "Pastikan checkpoint tersimpan atau isi CHECKPOINT_PATH_OVERRIDE."
        )

    raise RuntimeError(
        "Ditemukan beberapa checkpoint FrameMean dengan seed yang sama. "
        "Set CHECKPOINT_PATH_OVERRIDE:\n"
        + "\n".join(str(p) for p in recursive_matches)
    )


CHECKPOINT_PATH = resolve_checkpoint(
    RUN_SEED
)

# ============================================================
# HELD-OUT DATASET
# ============================================================
BENCHMARK_ROOT = Path(
    "/content/drive/MyDrive/Benchmark"
)

EXISTING_CACHE_DIR = (
    MODEL_ROOT
    / "new_saved_dataset"
)

HELDOUT_CACHE_DIR = (
    MODEL_ROOT
    / "heldout_cache_rgb64_v2"
)

HELDOUT_SPECS = {
    "RWF": {
        "raw_dir": BENCHMARK_ROOT / "RWF",
        "cache_name": "RWF",
        "cache_dir": EXISTING_CACHE_DIR,
        "nonviolence_dir": None,
        "violence_dir": None,
    },
    "UCF": {
        "raw_dir": BENCHMARK_ROOT / "UCF",
        "cache_name": "UCF",
        "cache_dir": HELDOUT_CACHE_DIR,
        "nonviolence_dir": None,
        "violence_dir": None,
    },
    "BV": {
        "raw_dir": BENCHMARK_ROOT / "BV",
        "cache_name": "BV",
        "cache_dir": HELDOUT_CACHE_DIR,
        "nonviolence_dir": None,
        "violence_dir": None,
    },
}

NONVIOLENCE_ALIASES = (
    "NonViolence",
    "Non-Violence",
    "Non_Violence",
    "Nonviolent",
    "NonViolent",
    "NonFight",
    "Non-Fight",
    "Non_Fight",
    "Normal",
    "NormalVideos",
    "Normal_Videos",
)

VIOLENCE_ALIASES = (
    "Violence",
    "Violent",
    "Fight",
    "Fighting",
    "FightVideos",
    "ViolentVideos",
    "Abnormal",
)

# ============================================================
# MODEL / PREPROCESSING
# ============================================================
CLASS_NAMES = ["NonViolence", "Violence"]
NUM_CLASSES = 2

SEQUENCE_LENGTH = 16
RAW_HEIGHT = 64
RAW_WIDTH = 64
BACKBONE_INPUT_SIZE = 224

# ============================================================
# FRAME DIVERSITY — HARUS SAMA UNTUK SEMUA SEED
# ============================================================
K_VALUES = [1, 2, 4, 8, 16]
NUM_RANDOM_REPEATS = 10
SAMPLING_BASE_SEED = 20260810

REFERENCE_BATCH_SIZE = 4
NUM_WORKERS = 2
USE_AMP = False

# K=16 feature-cache harus mereproduksi direct FrameMean evaluation.
K16_PROBABILITY_ATOL = 1e-5
K16_METRIC_ATOL = 1e-5

# Held-out raw cache behavior.
PREFER_LEGACY_CACHE = True
LEGACY_CACHE_COLOR_ORDER = "BGR"
AUTO_SCALE_LEGACY_CACHE = True
ALLOW_BUILD_CACHE = True

# Per-seed feature cache karena backbone features bergantung pada checkpoint.
EVAL_OUTPUT_DIR = (
    MODEL_ROOT
    / "HeldOut_Evaluation"
    / "FrameMean_FrameDiversity"
    / f"seed_{RUN_SEED}"
)

FEATURE_CACHE_DIR = (
    EVAL_OUTPUT_DIR
    / "frame_feature_cache"
)

EVAL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FEATURE_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

HELDOUT_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("RUN_SEED          :", RUN_SEED)
print("CHECKPOINT_PATH   :", CHECKPOINT_PATH)
print("K_VALUES          :", K_VALUES)
print("RANDOM_REPEATS    :", NUM_RANDOM_REPEATS)
print("SAMPLING_BASE_SEED:", SAMPLING_BASE_SEED)
print("OUTPUT            :", EVAL_OUTPUT_DIR)

RUN_SEED          : 123
CHECKPOINT_PATH   : /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/EffTempNet_FrameMean_NoLS_123Seeds_v1/frame_mean/seed_123/best.pt
K_VALUES          : [1, 2, 4, 8, 16]
RANDOM_REPEATS    : 10
SAMPLING_BASE_SEED: 20260810
OUTPUT            : /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/FrameMean_FrameDiversity/seed_123


In [ ]:
def set_global_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id: int):
    worker_seed = (
        torch.initial_seed()
        % (2**32)
    )

    np.random.seed(worker_seed)
    random.seed(worker_seed)


def safe_torch_load(
    path: Path,
    map_location,
):
    try:
        return torch.load(
            path,
            map_location=map_location,
            weights_only=False,
        )

    except TypeError:
        return torch.load(
            path,
            map_location=map_location,
        )


def assert_finite_tensor(
    name: str,
    tensor: torch.Tensor,
):
    if not torch.isfinite(
        tensor
    ).all():
        raise RuntimeError(
            f"{name} contains NaN/Inf."
        )


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

set_global_seed(
    RUN_SEED
)

print("DEVICE:", DEVICE)

DEVICE: cuda


## 2. FrameMean model — identik dengan training

Arsitektur:

```text
16 frames
→ shared EfficientFormerV2-S0 per frame
→ per-frame spatial feature
→ feature dropout
→ mean across temporal dimension
→ classifier
```

Tidak ada BiLSTM dan tidak ada temporal attention.

Eksperimen frame-diversity tidak melakukan retraining dan tidak mengubah
classifier. Hanya subset distinct frame-features yang masuk ke operator mean.

In [ ]:
class EfficientFormerFrameMean(nn.Module):
    VALID_VARIANTS = {
        "frame_mean"
    }

    def __init__(
        self,
        variant: str = "frame_mean",
        backbone_name: str = "efficientformerv2_s0",
        num_classes: int = NUM_CLASSES,
        pretrained: bool = False,
    ):
        super().__init__()

        if variant not in self.VALID_VARIANTS:
            raise ValueError(
                f"Unknown variant: {variant}"
            )

        self.variant = variant

        self.backbone = create_model(
            backbone_name,
            pretrained=pretrained,
            num_classes=0,
            in_chans=3,
        )

        self.feature_dim = int(
            self.backbone.num_features
        )

        self.feature_dropout = nn.Dropout(
            0.2
        )

        self.classifier = nn.Sequential(
            nn.Linear(
                self.feature_dim,
                256,
            ),
            nn.ReLU(),
            nn.Dropout(0.25),

            nn.Linear(
                256,
                128,
            ),
            nn.ReLU(),
            nn.Dropout(0.25),

            nn.Linear(
                128,
                64,
            ),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(
                64,
                32,
            ),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(
                32,
                num_classes,
            ),
        )

        self._initialize_classifier()

        self.lstm = None
        self.attention = None

    def _initialize_classifier(
        self,
    ):
        for module in self.classifier.modules():
            if isinstance(
                module,
                nn.Linear,
            ):
                nn.init.kaiming_uniform_(
                    module.weight,
                    nonlinearity="relu",
                )

                if module.bias is not None:
                    nn.init.zeros_(
                        module.bias
                    )

    def forward(
        self,
        clip: torch.Tensor,
    ):
        if clip.ndim != 5:
            raise ValueError(
                f"Expected [B,T,C,H,W], got {tuple(clip.shape)}"
            )

        (
            batch_size,
            seq_len,
            channels,
            height,
            width,
        ) = clip.shape

        if seq_len != SEQUENCE_LENGTH:
            raise ValueError(
                f"Expected T={SEQUENCE_LENGTH}, got T={seq_len}"
            )

        if channels != 3:
            raise ValueError(
                f"Expected 3 channels, got {channels}"
            )

        x = clip.reshape(
            batch_size * seq_len,
            channels,
            height,
            width,
        )

        x = F.interpolate(
            x,
            size=(
                BACKBONE_INPUT_SIZE,
                BACKBONE_INPUT_SIZE,
            ),
            mode="bilinear",
            align_corners=False,
        )

        features = self.backbone(
            x
        )

        if features.ndim == 4:
            features = (
                F.adaptive_avg_pool2d(
                    features,
                    output_size=1,
                )
                .flatten(1)
            )

        elif features.ndim != 2:
            features = (
                features
                .flatten(1)
            )

        features = features.view(
            batch_size,
            seq_len,
            -1,
        )

        features = self.feature_dropout(
            features
        )

        context = features.mean(
            dim=1
        )

        return self.classifier(
            context
        )

## 3. Resolve / build shared held-out 16-frame cache

RWF menggunakan cache eksperimen yang sudah ada bila tersedia.

UCF dan Bus Violence menggunakan shared held-out cache yang sama dengan
evaluator model lain.

Jika cache RGB-v2 belum ada, notebook dapat membangunnya dari raw clips:
- 16 posisi uniform;
- OpenCV BGR → RGB;
- resize 64×64;
- float32 [0,1].

Cache raw-video **shared antar-seed**; yang seed-specific hanya feature cache
hasil backbone FrameMean.

In [ ]:
VIDEO_EXTENSIONS = {
    ".mp4", ".avi", ".mov", ".mkv",
    ".mpeg", ".mpg", ".webm"
}

def direct_subdirs(root: Path) -> Dict[str, Path]:
    if not root.exists():
        return {}
    return {
        p.name: p
        for p in root.iterdir()
        if p.is_dir()
    }

def resolve_one_class_dir(
    root: Path,
    explicit_name: Optional[str],
    aliases: Tuple[str, ...],
    canonical_name: str,
) -> Path:
    if explicit_name is not None:
        candidate = root / explicit_name
        if not candidate.exists():
            raise FileNotFoundError(
                f"Folder {canonical_name} tidak ditemukan: {candidate}"
            )
        return candidate

    subdirs = direct_subdirs(root)
    lower_map = {
        name.lower(): path
        for name, path in subdirs.items()
    }

    matches = []
    for alias in aliases:
        candidate = lower_map.get(alias.lower())
        if candidate is not None and candidate not in matches:
            matches.append(candidate)

    if len(matches) != 1:
        raise RuntimeError(
            f"Tidak dapat menentukan folder kelas {canonical_name} "
            f"secara aman pada {root}.\n"
            f"Matches: {[p.name for p in matches]}\n"
            f"Available: {sorted(subdirs.keys())}\n"
            "Isi nonviolence_dir/violence_dir di HELDOUT_SPECS."
        )

    return matches[0]

def list_videos_for_spec(
    raw_dir: Path,
    nonviolence_dir: Optional[str],
    violence_dir: Optional[str],
) -> List[Tuple[Path, int]]:
    if not raw_dir.exists():
        raise FileNotFoundError(
            f"Raw held-out directory tidak ditemukan: {raw_dir}"
        )

    class_dirs = {
        0: resolve_one_class_dir(
            raw_dir,
            nonviolence_dir,
            NONVIOLENCE_ALIASES,
            "NonViolence",
        ),
        1: resolve_one_class_dir(
            raw_dir,
            violence_dir,
            VIOLENCE_ALIASES,
            "Violence",
        ),
    }

    records = []

    for label, class_dir in class_dirs.items():
        videos = sorted(
            p for p in class_dir.rglob("*")
            if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
        )

        if not videos:
            raise RuntimeError(
                f"Tidak ada video pada folder kelas: {class_dir}"
            )

        print(
            f"{CLASS_NAMES[label]:12s}: {len(videos):5d} clips | {class_dir}"
        )

        records.extend(
            (p, label)
            for p in videos
        )

    labels = np.asarray(
        [label for _, label in records],
        dtype=np.int64,
    )

    if set(np.unique(labels).tolist()) != {0, 1}:
        raise RuntimeError(
            "Held-out dataset harus mempunyai kedua kelas: "
            "NonViolence dan Violence."
        )

    return records

def extract_uniform_rgb_frames(
    video_path: Path,
    sequence_length: int = SEQUENCE_LENGTH,
    height: int = RAW_HEIGHT,
    width: int = RAW_WIDTH,
) -> Optional[np.ndarray]:
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        return None

    frame_count = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    if frame_count <= 0:
        cap.release()
        return None

    frame_positions = np.linspace(
        0,
        max(frame_count - 1, 0),
        sequence_length,
    ).astype(int)

    frames = []

    for pos in frame_positions:
        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(pos),
        )

        ok, frame_bgr = cap.read()

        if not ok:
            break

        frame_rgb = cv2.cvtColor(
            frame_bgr,
            cv2.COLOR_BGR2RGB,
        )

        frame_rgb = cv2.resize(
            frame_rgb,
            (width, height),
            interpolation=cv2.INTER_LINEAR,
        )

        frames.append(
            frame_rgb.astype(np.float32) / 255.0
        )

    cap.release()

    if not frames:
        return None

    while len(frames) < sequence_length:
        frames.append(frames[-1].copy())

    return np.stack(
        frames[:sequence_length],
        axis=0,
    ).astype(np.float32)

def build_rgb_cache(
    raw_dir: Path,
    cache_dir: Path,
    cache_name: str,
    nonviolence_dir: Optional[str],
    violence_dir: Optional[str],
):
    cache_dir.mkdir(parents=True, exist_ok=True)

    feature_path = cache_dir / f"{cache_name}_rgb64_v2_features.npy"
    label_path = cache_dir / f"{cache_name}_rgb64_v2_labels.npy"
    paths_path = cache_dir / f"{cache_name}_rgb64_v2_paths.json"

    records = list_videos_for_spec(
        raw_dir,
        nonviolence_dir,
        violence_dir,
    )

    # Gunakan temporary memmap agar tidak menyimpan semua clip di RAM.
    n_requested = len(records)

    temp_path = cache_dir / f".{cache_name}_tmp_features.npy"

    if temp_path.exists():
        temp_path.unlink()

    temp = np.lib.format.open_memmap(
        temp_path,
        mode="w+",
        dtype=np.float32,
        shape=(
            n_requested,
            SEQUENCE_LENGTH,
            RAW_HEIGHT,
            RAW_WIDTH,
            3,
        ),
    )

    labels_buffer = np.empty(
        n_requested,
        dtype=np.int64,
    )

    valid_paths = []
    failed_paths = []
    valid_count = 0

    for video_path, label in tqdm(
        records,
        desc=f"Build cache {cache_name}",
        unit="video",
    ):
        clip = extract_uniform_rgb_frames(video_path)

        if clip is None:
            failed_paths.append(str(video_path))
            continue

        temp[valid_count] = clip
        labels_buffer[valid_count] = int(label)
        valid_paths.append(str(video_path))
        valid_count += 1

    temp.flush()
    del temp

    if valid_count == 0:
        if temp_path.exists():
            temp_path.unlink()
        raise RuntimeError(
            f"Semua video gagal didekode untuk {cache_name}."
        )

    # Compact hanya sample valid ke final .npy.
    temp_read = np.load(
        temp_path,
        mmap_mode="r",
    )

    final = np.lib.format.open_memmap(
        feature_path,
        mode="w+",
        dtype=np.float32,
        shape=(
            valid_count,
            SEQUENCE_LENGTH,
            RAW_HEIGHT,
            RAW_WIDTH,
            3,
        ),
    )

    chunk_size = 16

    for start in range(0, valid_count, chunk_size):
        end = min(start + chunk_size, valid_count)
        final[start:end] = temp_read[start:end]

    final.flush()
    del final, temp_read

    np.save(
        label_path,
        labels_buffer[:valid_count],
    )

    with open(
        paths_path,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            {
                "paths": valid_paths,
                "failed": failed_paths,
                "n_requested": n_requested,
                "n_valid": valid_count,
                "color_order": "RGB",
                "sequence_length": SEQUENCE_LENGTH,
                "height": RAW_HEIGHT,
                "width": RAW_WIDTH,
            },
            f,
            indent=2,
        )

    if temp_path.exists():
        temp_path.unlink()

    print(
        f"Saved {feature_path} | "
        f"shape=({valid_count},{SEQUENCE_LENGTH},{RAW_HEIGHT},{RAW_WIDTH},3)"
    )
    print("Failed videos:", len(failed_paths))

    return {
        "feature_path": feature_path,
        "label_path": label_path,
        "paths_path": paths_path,
        "color_order": "RGB",
        "cache_source": "built_rgb64_v2",
    }

def resolve_heldout_store(
    dataset_name: str,
    spec: Dict,
) -> Dict:
    raw_dir = Path(spec["raw_dir"])
    cache_dir = Path(spec["cache_dir"])
    cache_name = str(spec["cache_name"])

    rgb_feature = cache_dir / f"{cache_name}_rgb64_v2_features.npy"
    rgb_label = cache_dir / f"{cache_name}_rgb64_v2_labels.npy"
    rgb_paths = cache_dir / f"{cache_name}_rgb64_v2_paths.json"

    legacy_feature = cache_dir / f"{cache_name}_features.npy"
    legacy_label = cache_dir / f"{cache_name}_labels.npy"
    legacy_paths = cache_dir / f"{cache_name}_paths.json"

    # Sama dengan training: RGB-v2 diprioritaskan; legacy dipakai jika v2 tidak ada.
    if rgb_feature.exists() and rgb_label.exists():
        print(f"✅ {dataset_name}: RGB-v2 cache")
        return {
            "feature_path": rgb_feature,
            "label_path": rgb_label,
            "paths_path": rgb_paths if rgb_paths.exists() else None,
            "color_order": "RGB",
            "cache_source": "rgb64_v2",
        }

    if (
        PREFER_LEGACY_CACHE
        and legacy_feature.exists()
        and legacy_label.exists()
    ):
        print(
            f"✅ {dataset_name}: legacy cache "
            f"({LEGACY_CACHE_COLOR_ORDER}→RGB saat loading)"
        )
        return {
            "feature_path": legacy_feature,
            "label_path": legacy_label,
            "paths_path": legacy_paths if legacy_paths.exists() else None,
            "color_order": LEGACY_CACHE_COLOR_ORDER,
            "cache_source": "legacy",
        }

    if not ALLOW_BUILD_CACHE:
        raise FileNotFoundError(
            f"Cache {dataset_name} tidak ada dan ALLOW_BUILD_CACHE=False."
        )

    if not raw_dir.exists():
        raise FileNotFoundError(
            f"Cache dan raw folder tidak ditemukan untuk {dataset_name}.\n"
            f"Raw expected: {raw_dir}"
        )

    print(
        f"ℹ️ {dataset_name}: cache belum ada, dibuat dari {raw_dir}"
    )

    return build_rgb_cache(
        raw_dir=raw_dir,
        cache_dir=cache_dir,
        cache_name=cache_name,
        nonviolence_dir=spec.get("nonviolence_dir"),
        violence_dir=spec.get("violence_dir"),
    )

In [ ]:
IMAGENET_MEAN = torch.tensor(
    [
        0.485,
        0.456,
        0.406,
    ],
    dtype=torch.float32,
).view(
    1,
    3,
    1,
    1,
)

IMAGENET_STD = torch.tensor(
    [
        0.229,
        0.224,
        0.225,
    ],
    dtype=torch.float32,
).view(
    1,
    3,
    1,
    1,
)


class Full16FrameHeldOutDataset(
    Dataset
):
    def __init__(
        self,
        dataset_name: str,
        feature_path: Path,
        label_path: Path,
        color_order: str,
    ):
        self.dataset_name = str(
            dataset_name
        )

        self.features = np.load(
            feature_path,
            mmap_mode="r",
        )

        self.labels = np.load(
            label_path,
            mmap_mode="r",
        ).astype(
            np.int64
        )

        self.color_order = str(
            color_order
        ).upper()

        if len(
            self.features
        ) != len(
            self.labels
        ):
            raise ValueError(
                f"{dataset_name}: feature/label length mismatch."
            )

        expected_shape = (
            SEQUENCE_LENGTH,
            RAW_HEIGHT,
            RAW_WIDTH,
            3,
        )

        if (
            self.features.ndim != 5
            or tuple(
                self.features.shape[
                    1:
                ]
            )
            != expected_shape
        ):
            raise ValueError(
                f"{dataset_name}: cache shape {self.features.shape}; "
                f"expected [N,{SEQUENCE_LENGTH},{RAW_HEIGHT},{RAW_WIDTH},3]."
            )

        unique_labels = set(
            np.unique(
                self.labels
            ).tolist()
        )

        if unique_labels != {
            0,
            1,
        }:
            raise ValueError(
                f"{dataset_name}: held-out dataset harus mempunyai "
                f"dua kelas {{0,1}}, found {sorted(unique_labels)}."
            )

        if self.color_order not in {
            "RGB",
            "BGR",
        }:
            raise ValueError(
                f"Unsupported color_order={self.color_order}"
            )

    def __len__(
        self,
    ):
        return len(
            self.labels
        )

    def __getitem__(
        self,
        index: int,
    ):
        frames = np.asarray(
            self.features[
                index
            ],
            dtype=np.float32,
        )

        label = int(
            self.labels[
                index
            ]
        )

        if not np.isfinite(
            frames
        ).all():
            raise ValueError(
                f"{self.dataset_name}: NaN/Inf sample={index}"
            )

        frame_min = float(
            frames.min()
        )

        frame_max = float(
            frames.max()
        )

        if (
            AUTO_SCALE_LEGACY_CACHE
            and frame_min >= 0.0
            and frame_max > 1.5
        ):
            if frame_max <= 255.0 + 1e-3:
                frames = (
                    frames
                    / 255.0
                )

            else:
                raise ValueError(
                    f"Invalid cache range [{frame_min},{frame_max}]"
                )

        if (
            float(
                frames.min()
            )
            < -1e-4
            or float(
                frames.max()
            )
            > 1.0001
        ):
            raise ValueError(
                f"Cache must be [0,1] after scaling; sample={index}"
            )

        if self.color_order == "BGR":
            frames = frames[
                ...,
                [
                    2,
                    1,
                    0,
                ],
            ]

        clip = torch.from_numpy(
            np.ascontiguousarray(
                frames
            )
        ).permute(
            0,
            3,
            1,
            2,
        )

        clip = (
            clip
            - IMAGENET_MEAN
        ) / IMAGENET_STD

        return (
            clip,
            torch.tensor(
                label,
                dtype=torch.long,
            ),
            int(index),
        )


def make_reference_loader(
    dataset_name: str,
    store: Dict,
):
    dataset = Full16FrameHeldOutDataset(
        dataset_name=dataset_name,
        feature_path=store[
            "feature_path"
        ],
        label_path=store[
            "label_path"
        ],
        color_order=store[
            "color_order"
        ],
    )

    generator = torch.Generator()

    generator.manual_seed(
        20260810
    )

    loader = DataLoader(
        dataset,
        batch_size=REFERENCE_BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=(
            DEVICE.type
            == "cuda"
        ),
        worker_init_fn=seed_worker,
        generator=generator,
        persistent_workers=(
            NUM_WORKERS > 0
        ),
    )

    return (
        dataset,
        loader,
    )


def load_paths_if_available(
    paths_path: Optional[Path],
    expected_n: int,
) -> Optional[List[str]]:
    if paths_path is None:
        return None

    path = Path(
        paths_path
    )

    if not path.exists():
        return None

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as file:
        payload = json.load(
            file
        )

    paths = payload.get(
        "paths"
    )

    if not isinstance(
        paths,
        list,
    ):
        return None

    if len(
        paths
    ) != expected_n:
        print(
            f"⚠️ paths count {len(paths)} != {expected_n}; "
            "fallback sample_key akan dipakai."
        )

        return None

    return [
        str(p)
        for p
        in paths
    ]

## 4. Load dan audit `best.pt`

Checkpoint harus FrameMean pada seed yang dipilih.

Audit:
- `variant = frame_mean` bila metadata tersedia;
- `seed` cocok bila metadata tersedia;
- `sequence_length = 16`;
- `strict=True`;
- no LSTM;
- no temporal attention;
- dummy forward menghasilkan `[B,2]`.

Semua parameter dibekukan karena notebook ini inference-only.

In [ ]:
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint tidak ditemukan: {CHECKPOINT_PATH}"
    )

checkpoint = safe_torch_load(
    CHECKPOINT_PATH,
    map_location="cpu",
)

if not isinstance(
    checkpoint,
    dict,
):
    raise TypeError(
        "Checkpoint bukan dictionary."
    )

if (
    "model_state_dict"
    not in checkpoint
):
    raise KeyError(
        "Checkpoint tidak mempunyai model_state_dict."
    )

checkpoint_config = checkpoint.get(
    "config",
    {},
)

expected_config = {
    "variant": VARIANT,
    "seed": RUN_SEED,
    "sequence_length": SEQUENCE_LENGTH,
}

mismatches = []

for key, expected in expected_config.items():
    if key in checkpoint_config:
        actual = checkpoint_config[
            key
        ]

        if actual != expected:
            mismatches.append(
                (
                    key,
                    actual,
                    expected,
                )
            )

if mismatches:
    raise RuntimeError(
        "Checkpoint config mismatch:\n"
        + "\n".join(
            f"- {key}: actual={actual}, expected={expected}"
            for (
                key,
                actual,
                expected,
            )
            in mismatches
        )
    )

model = EfficientFormerFrameMean(
    variant=VARIANT,
    pretrained=False,
).to(
    DEVICE
)

load_result = model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ],
    strict=True,
)

model.eval()

for parameter in model.parameters():
    parameter.requires_grad_(
        False
    )

if model.lstm is not None:
    raise RuntimeError(
        "FrameMean evaluator tidak boleh mempunyai LSTM."
    )

if model.attention is not None:
    raise RuntimeError(
        "FrameMean evaluator tidak boleh mempunyai temporal attention."
    )

with torch.inference_mode():
    dummy = torch.zeros(
        1,
        SEQUENCE_LENGTH,
        3,
        RAW_HEIGHT,
        RAW_WIDTH,
        device=DEVICE,
    )

    dummy_logits = model(
        dummy
    )

if tuple(
    dummy_logits.shape
) != (
    1,
    NUM_CLASSES,
):
    raise RuntimeError(
        f"Unexpected output shape: {tuple(dummy_logits.shape)}"
    )

assert_finite_tensor(
    "dummy_logits",
    dummy_logits,
)

print("✅ Checkpoint audit PASS")
print("Checkpoint :", CHECKPOINT_PATH)
print("Epoch      :", checkpoint.get("epoch", "unknown"))
print("Best val F1:", checkpoint.get("best_score", "unknown"))
print("Load result:", load_result)
print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    ),
)

del (
    dummy,
    dummy_logits,
)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

✅ Checkpoint audit PASS
Checkpoint : /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/EffTempNet_FrameMean_NoLS_123Seeds_v1/frame_mean/seed_123/best.pt
Epoch      : 17
Best val F1: 0.8565373528118434
Load result: <All keys matched successfully>
Trainable parameters: 0


## 5. Resolve dan audit RWF, UCF, dan Bus Violence

In [ ]:
heldout_stores = {}
dataset_audit_rows = []

for (
    dataset_name,
    spec,
) in HELDOUT_SPECS.items():
    print(
        "\n"
        + "=" * 90
    )

    print(
        "DATASET:",
        dataset_name,
    )

    print(
        "=" * 90
    )

    store = resolve_heldout_store(
        dataset_name,
        spec,
    )

    dataset = Full16FrameHeldOutDataset(
        dataset_name=dataset_name,
        feature_path=store[
            "feature_path"
        ],
        label_path=store[
            "label_path"
        ],
        color_order=store[
            "color_order"
        ],
    )

    labels = np.asarray(
        dataset.labels,
        dtype=np.int64,
    )

    counts = np.bincount(
        labels,
        minlength=2,
    )

    heldout_stores[
        dataset_name
    ] = store

    dataset_audit_rows.append(
        {
            "dataset": dataset_name,
            "n_total": int(
                len(dataset)
            ),
            "n_nonviolence": int(
                counts[0]
            ),
            "n_violence": int(
                counts[1]
            ),
            "feature_path": str(
                store[
                    "feature_path"
                ]
            ),
            "label_path": str(
                store[
                    "label_path"
                ]
            ),
            "cache_source": store[
                "cache_source"
            ],
            "color_order": store[
                "color_order"
            ],
        }
    )

    print(
        f"✅ {dataset_name}: "
        f"N={len(dataset)}, "
        f"NV={counts[0]}, "
        f"V={counts[1]}, "
        f"cache={store['cache_source']}"
    )

dataset_audit_df = pd.DataFrame(
    dataset_audit_rows
)

dataset_audit_path = (
    EVAL_OUTPUT_DIR
    / "heldout_dataset_audit.csv"
)

dataset_audit_df.to_csv(
    dataset_audit_path,
    index=False,
)

display(
    dataset_audit_df
)


DATASET: RWF
✅ RWF: legacy cache (BGR→RGB saat loading)
✅ RWF: N=1998, NV=1000, V=998, cache=legacy

DATASET: UCF
✅ UCF: RGB-v2 cache
✅ UCF: N=100, NV=50, V=50, cache=rgb64_v2

DATASET: BV
✅ BV: RGB-v2 cache
✅ BV: N=1400, NV=700, V=700, cache=rgb64_v2


,dataset,n_total,n_nonviolence,n_violence,feature_path,label_path,cache_source,color_order
0,RWF,1998,1000,998,/content/drive/MyDrive/IVS/ViolenceDetection/M...,/content/drive/MyDrive/IVS/ViolenceDetection/M...,legacy,BGR
1,UCF,100,50,50,/content/drive/MyDrive/IVS/ViolenceDetection/M...,/content/drive/MyDrive/IVS/ViolenceDetection/M...,rgb64_v2,RGB
2,BV,1400,700,700,/content/drive/MyDrive/IVS/ViolenceDetection/M...,/content/drive/MyDrive/IVS/ViolenceDetection/M...,rgb64_v2,RGB


## 6. Metrics, direct evaluation, dan per-frame feature extraction

K=16 dari feature cache akan dibandingkan dengan direct `model(clips)` sebagai
validity gate sebelum K<16 dijalankan.

In [ ]:
def compute_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    p_violence: np.ndarray,
) -> Dict[str, float]:
    (
        precision_cls,
        recall_cls,
        f1_cls,
        support,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[
            0,
            1,
        ],
        zero_division=0,
    )

    result = {
        "n": int(
            len(
                y_true
            )
        ),
        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "precision_macro": float(
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
        "recall_macro": float(
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
        "f1_macro": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
        "precision_nonviolence": float(
            precision_cls[
                0
            ]
        ),
        "recall_nonviolence": float(
            recall_cls[
                0
            ]
        ),
        "f1_nonviolence": float(
            f1_cls[
                0
            ]
        ),
        "precision_violence": float(
            precision_cls[
                1
            ]
        ),
        "recall_violence": float(
            recall_cls[
                1
            ]
        ),
        "f1_violence": float(
            f1_cls[
                1
            ]
        ),
    }

    if len(
        np.unique(
            y_true
        )
    ) == 2:
        result[
            "roc_auc"
        ] = float(
            roc_auc_score(
                y_true,
                p_violence,
            )
        )

        result[
            "average_precision"
        ] = float(
            average_precision_score(
                y_true,
                p_violence,
            )
        )

    else:
        result[
            "roc_auc"
        ] = float(
            "nan"
        )

        result[
            "average_precision"
        ] = float(
            "nan"
        )

    return result


@torch.inference_mode()
def extract_frame_features(
    clips: torch.Tensor,
) -> torch.Tensor:
    model.eval()

    (
        batch_size,
        seq_len,
        channels,
        height,
        width,
    ) = clips.shape

    if seq_len != SEQUENCE_LENGTH:
        raise ValueError(
            f"Expected T={SEQUENCE_LENGTH}, got T={seq_len}"
        )

    x = clips.reshape(
        batch_size
        * seq_len,
        channels,
        height,
        width,
    )

    x = F.interpolate(
        x,
        size=(
            BACKBONE_INPUT_SIZE,
            BACKBONE_INPUT_SIZE,
        ),
        mode="bilinear",
        align_corners=False,
    )

    features = model.backbone(
        x
    )

    if features.ndim == 4:
        features = (
            F.adaptive_avg_pool2d(
                features,
                output_size=1,
            )
            .flatten(1)
        )

    elif features.ndim != 2:
        features = features.flatten(
            1
        )

    features = features.view(
        batch_size,
        seq_len,
        -1,
    )

    features = model.feature_dropout(
        features
    )

    return features


@torch.inference_mode()
def evaluate_direct_dataset(
    dataset_name: str,
    store: Dict,
):
    (
        dataset,
        loader,
    ) = make_reference_loader(
        dataset_name,
        store,
    )

    source_paths = load_paths_if_available(
        store.get(
            "paths_path"
        ),
        expected_n=len(
            dataset
        ),
    )

    all_indices = []
    all_labels = []
    all_predictions = []
    all_probabilities = []

    model.eval()

    for (
        clips,
        labels,
        indices,
    ) in tqdm(
        loader,
        desc=f"Direct {dataset_name}",
        leave=False,
    ):
        clips = clips.to(
            DEVICE,
            non_blocking=True,
        )

        logits = model(
            clips
        )

        probabilities = torch.softmax(
            logits.float(),
            dim=1,
        )

        predictions = torch.argmax(
            probabilities,
            dim=1,
        )

        all_indices.extend(
            np.asarray(
                indices
            ).tolist()
        )

        all_labels.extend(
            labels.numpy().tolist()
        )

        all_predictions.extend(
            predictions.cpu().numpy().tolist()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy().tolist()
        )

    sample_indices = np.asarray(
        all_indices,
        dtype=np.int64,
    )

    y_true = np.asarray(
        all_labels,
        dtype=np.int64,
    )

    y_pred = np.asarray(
        all_predictions,
        dtype=np.int64,
    )

    probs = np.asarray(
        all_probabilities,
        dtype=np.float64,
    )

    if source_paths is None:
        sample_keys = [
            f"{dataset_name}::{int(index)}"
            for index
            in sample_indices
        ]

        video_paths = [
            ""
            for _
            in sample_indices
        ]

    else:
        video_paths = [
            source_paths[
                int(index)
            ]
            for index
            in sample_indices
        ]

        sample_keys = [
            Path(path).as_posix()
            for path
            in video_paths
        ]

    prediction_df = pd.DataFrame(
        {
            "dataset": dataset_name,
            "variant": VARIANT,
            "seed": RUN_SEED,
            "k_unique": 16,
            "repeat": 0,
            "sample_index": sample_indices,
            "sample_key": sample_keys,
            "video_path": video_paths,
            "selected_indices_0based": json.dumps(
                list(
                    range(
                        SEQUENCE_LENGTH
                    )
                )
            ),
            "y_true": y_true,
            "y_pred": y_pred,
            "is_correct": (
                y_true
                == y_pred
            ).astype(
                np.int64
            ),
            "p_nonviolence": probs[
                :,
                0
            ],
            "p_violence": probs[
                :,
                1
            ],
        }
    )

    if prediction_df[
        "sample_key"
    ].duplicated().any():
        raise RuntimeError(
            f"Duplicate sample_key pada {dataset_name}."
        )

    metrics = compute_metrics(
        y_true,
        y_pred,
        probs[
            :,
            1
        ],
    )

    return (
        metrics,
        prediction_df,
    )

## 7. Build / reuse per-seed frame-feature cache

In [ ]:
@torch.inference_mode()
def build_or_load_feature_cache(
    dataset_name: str,
    store: Dict,
):
    feature_cache_path = (
        FEATURE_CACHE_DIR
        / f"{dataset_name}_frame_features.npy"
    )

    label_cache_path = (
        FEATURE_CACHE_DIR
        / f"{dataset_name}_labels.npy"
    )

    index_cache_path = (
        FEATURE_CACHE_DIR
        / f"{dataset_name}_indices.npy"
    )

    meta_path = (
        FEATURE_CACHE_DIR
        / f"{dataset_name}_meta.json"
    )

    can_reuse = (
        feature_cache_path.exists()
        and label_cache_path.exists()
        and index_cache_path.exists()
        and meta_path.exists()
    )

    if can_reuse:
        with open(
            meta_path,
            "r",
            encoding="utf-8",
        ) as file:
            meta = json.load(
                file
            )

        if (
            meta.get(
                "version"
            )
            == "heldout_frame_diversity_v1"
            and meta.get(
                "checkpoint_seed"
            )
            == RUN_SEED
            and meta.get(
                "checkpoint_path"
            )
            == str(
                CHECKPOINT_PATH
            )
            and meta.get(
                "reference_batch_size"
            )
            == REFERENCE_BATCH_SIZE
        ):
            print(
                f"♻️ Reuse feature cache: {dataset_name}"
            )

            return {
                "feature_path": feature_cache_path,
                "label_path": label_cache_path,
                "index_path": index_cache_path,
            }

    (
        dataset,
        loader,
    ) = make_reference_loader(
        dataset_name,
        store,
    )

    feature_batches = []
    label_batches = []
    index_batches = []

    model.eval()

    for (
        clips,
        labels,
        indices,
    ) in tqdm(
        loader,
        desc=f"Extract features {dataset_name}",
    ):
        clips = clips.to(
            DEVICE,
            non_blocking=True,
        )

        frame_features = extract_frame_features(
            clips
        )

        assert_finite_tensor(
            f"{dataset_name}_frame_features",
            frame_features,
        )

        feature_batches.append(
            frame_features.cpu().numpy().astype(
                np.float32
            )
        )

        label_batches.append(
            labels.numpy().astype(
                np.int64
            )
        )

        index_batches.append(
            np.asarray(
                indices
            ).astype(
                np.int64
            )
        )

    frame_features_np = np.concatenate(
        feature_batches,
        axis=0,
    )

    labels_np = np.concatenate(
        label_batches,
        axis=0,
    )

    indices_np = np.concatenate(
        index_batches,
        axis=0,
    )

    if frame_features_np.shape[
        :2
    ] != (
        len(
            labels_np
        ),
        SEQUENCE_LENGTH,
    ):
        raise RuntimeError(
            f"Invalid feature cache shape: {frame_features_np.shape}"
        )

    np.save(
        feature_cache_path,
        frame_features_np,
    )

    np.save(
        label_cache_path,
        labels_np,
    )

    np.save(
        index_cache_path,
        indices_np,
    )

    with open(
        meta_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            {
                "version": "heldout_frame_diversity_v1",
                "dataset": dataset_name,
                "checkpoint_seed": RUN_SEED,
                "checkpoint_path": str(
                    CHECKPOINT_PATH
                ),
                "reference_batch_size": REFERENCE_BATCH_SIZE,
                "shape": list(
                    frame_features_np.shape
                ),
                "dtype": "float32",
            },
            file,
            indent=2,
        )

    print(
        f"✅ Saved feature cache {dataset_name}: "
        f"{frame_features_np.shape}"
    )

    del (
        frame_features_np,
        labels_np,
        indices_np,
        feature_batches,
        label_batches,
        index_batches,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "feature_path": feature_cache_path,
        "label_path": label_cache_path,
        "index_path": index_cache_path,
    }


FEATURE_STORES = {}

for (
    dataset_name,
    store,
) in heldout_stores.items():
    FEATURE_STORES[
        dataset_name
    ] = build_or_load_feature_cache(
        dataset_name,
        store,
    )

print(
    "✅ Semua held-out frame-feature cache selesai."
)

Extract features RWF:   0%|          | 0/500 [00:00<?, ?it/s]

✅ Saved feature cache RWF: (1998, 16, 176)


Extract features UCF:   0%|          | 0/25 [00:00<?, ?it/s]

✅ Saved feature cache UCF: (100, 16, 176)


Extract features BV:   0%|          | 0/350 [00:00<?, ?it/s]

✅ Saved feature cache BV: (1400, 16, 176)
✅ Semua held-out frame-feature cache selesai.


## 8. Frame-diversity sampling

Untuk K<16, setiap clip:
1. 16 temporal positions dibagi menjadi K bin sama lebar;
2. satu posisi dipilih secara pseudo-random dari setiap bin;
3. selection seed bergantung pada:
   - fixed `SAMPLING_BASE_SEED`,
   - dataset,
   - sample index,
   - K,
   - repeat;
4. **tidak menggunakan RUN_SEED**.

Maka seed model 0/42/123 melihat frame-selection yang identik.

In [ ]:
def stable_dataset_code(
    dataset_name: str,
) -> int:
    return int(
        zlib.crc32(
            dataset_name.encode(
                "utf-8"
            )
        )
        & 0xFFFFFFFF
    )


def stratified_random_indices(
    dataset_name: str,
    source_idx: int,
    k: int,
    repeat_id: int,
) -> np.ndarray:
    if k not in K_VALUES:
        raise ValueError(
            f"K tidak valid: {k}"
        )

    if k == SEQUENCE_LENGTH:
        return np.arange(
            SEQUENCE_LENGTH,
            dtype=np.int64,
        )

    if (
        SEQUENCE_LENGTH
        % k
        != 0
    ):
        raise ValueError(
            f"K={k} harus membagi T={SEQUENCE_LENGTH}."
        )

    bin_width = (
        SEQUENCE_LENGTH
        // k
    )

    seed_sequence = (
        np.random.SeedSequence(
            [
                int(
                    SAMPLING_BASE_SEED
                ),
                int(
                    repeat_id
                ),
                int(
                    stable_dataset_code(
                        dataset_name
                    )
                ),
                int(
                    source_idx
                ),
                int(
                    k
                ),
            ]
        )
    )

    rng = np.random.default_rng(
        seed_sequence
    )

    selected = []

    for bin_id in range(
        k
    ):
        start = (
            bin_id
            * bin_width
        )

        stop = (
            start
            + bin_width
        )

        selected.append(
            int(
                rng.integers(
                    start,
                    stop,
                )
            )
        )

    selected = np.asarray(
        selected,
        dtype=np.int64,
    )

    if len(
        np.unique(
            selected
        )
    ) != k:
        raise RuntimeError(
            "Jumlah distinct positions tidak sama dengan K."
        )

    if not np.all(
        np.diff(
            selected
        )
        > 0
    ):
        raise RuntimeError(
            "Selected positions harus tetap kronologis."
        )

    return selected


# Quick deterministic audit.
for k in K_VALUES:
    a = stratified_random_indices(
        "RWF",
        source_idx=123,
        k=k,
        repeat_id=0,
    )

    b = stratified_random_indices(
        "RWF",
        source_idx=123,
        k=k,
        repeat_id=0,
    )

    if not np.array_equal(
        a,
        b,
    ):
        raise RuntimeError(
            "Sampling tidak deterministik."
        )

print(
    "✅ Frame-diversity sampling audit PASS"
)

✅ Frame-diversity sampling audit PASS


## 9. Evaluate K from cached FrameMean features

In [ ]:
@torch.inference_mode()
def evaluate_feature_dataset(
    dataset_name: str,
    k_unique: int,
    repeat_id: int,
):
    store = FEATURE_STORES[
        dataset_name
    ]

    raw_store = heldout_stores[
        dataset_name
    ]

    frame_features = np.load(
        store[
            "feature_path"
        ],
        mmap_mode="r",
    )

    labels = np.asarray(
        np.load(
            store[
                "label_path"
            ],
            mmap_mode="r",
        ),
        dtype=np.int64,
    )

    source_indices = np.asarray(
        np.load(
            store[
                "index_path"
            ],
            mmap_mode="r",
        ),
        dtype=np.int64,
    )

    source_paths = load_paths_if_available(
        raw_store.get(
            "paths_path"
        ),
        expected_n=len(
            labels
        ),
    )

    all_predictions = []
    all_probabilities = []
    selected_indices_strings = []

    model.eval()

    for start in range(
        0,
        len(
            labels
        ),
        REFERENCE_BATCH_SIZE,
    ):
        stop = min(
            start
            + REFERENCE_BATCH_SIZE,
            len(
                labels
            ),
        )

        batch_selected = []

        batch_selected_indices = []

        for row_id in range(
            start,
            stop,
        ):
            source_idx = int(
                source_indices[
                    row_id
                ]
            )

            selected_indices = (
                stratified_random_indices(
                    dataset_name=dataset_name,
                    source_idx=source_idx,
                    k=k_unique,
                    repeat_id=repeat_id,
                )
            )

            batch_selected_indices.append(
                selected_indices
            )

            batch_selected.append(
                np.asarray(
                    frame_features[
                        row_id,
                        selected_indices,
                        :,
                    ],
                    dtype=np.float32,
                )
            )

        selected_np = np.stack(
            batch_selected,
            axis=0,
        )

        selected = torch.from_numpy(
            np.ascontiguousarray(
                selected_np
            )
        ).to(
            DEVICE,
            dtype=torch.float32,
        )

        # Original V3 FrameMean frame-diversity behavior:
        # mean only over K distinct selected positions.
        # For K ∈ {1,2,4,8,16}, this is mathematically equivalent to
        # equal duplication of each selected feature to T=16 then mean.
        context = selected.mean(
            dim=1
        )

        logits = model.classifier(
            context
        )

        assert_finite_tensor(
            "frame_diversity_logits",
            logits,
        )

        probabilities = torch.softmax(
            logits.float(),
            dim=1,
        )

        predictions = torch.argmax(
            probabilities,
            dim=1,
        )

        all_predictions.extend(
            predictions.cpu().numpy().tolist()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy().tolist()
        )

        selected_indices_strings.extend(
            [
                json.dumps(
                    indices.tolist()
                )
                for indices
                in batch_selected_indices
            ]
        )

    y_pred = np.asarray(
        all_predictions,
        dtype=np.int64,
    )

    probs = np.asarray(
        all_probabilities,
        dtype=np.float64,
    )

    if source_paths is None:
        sample_keys = [
            f"{dataset_name}::{int(index)}"
            for index
            in source_indices
        ]

        video_paths = [
            ""
            for _
            in source_indices
        ]

    else:
        video_paths = [
            source_paths[
                int(index)
            ]
            for index
            in source_indices
        ]

        sample_keys = [
            Path(path).as_posix()
            for path
            in video_paths
        ]

    metrics = compute_metrics(
        labels,
        y_pred,
        probs[
            :,
            1
        ],
    )

    predictions_df = pd.DataFrame(
        {
            "dataset": dataset_name,
            "variant": VARIANT,
            "seed": RUN_SEED,
            "k_unique": k_unique,
            "repeat": repeat_id,
            "sample_index": source_indices,
            "sample_key": sample_keys,
            "video_path": video_paths,
            "selected_indices_0based": selected_indices_strings,
            "y_true": labels,
            "y_pred": y_pred,
            "is_correct": (
                labels
                == y_pred
            ).astype(
                np.int64
            ),
            "p_nonviolence": probs[
                :,
                0
            ],
            "p_violence": probs[
                :,
                1
            ],
        }
    )

    if predictions_df[
        "sample_key"
    ].duplicated().any():
        raise RuntimeError(
            f"Duplicate sample_key pada {dataset_name}, K={k_unique}, repeat={repeat_id}"
        )

    return (
        metrics,
        predictions_df,
    )

## 10. Validity gate — K=16 feature route harus mereproduksi direct FrameMean

Sebelum menjalankan K<16:
- direct `model(clips)` dievaluasi;
- feature-cache route K=16 dievaluasi;
- sample order harus sama;
- `y_true` harus sama;
- probability difference harus sangat kecil;
- metric difference harus sangat kecil.

Jika gate gagal, notebook berhenti.

In [ ]:
direct_metric_rows = []
direct_prediction_frames = []

k16_metric_rows = []
k16_prediction_frames = []

gate_rows = []

for (
    dataset_name,
    store,
) in heldout_stores.items():
    print(
        "\n"
        + "=" * 90
    )

    print(
        f"K=16 VALIDITY GATE | {dataset_name}"
    )

    print(
        "=" * 90
    )

    (
        direct_metrics,
        direct_predictions,
    ) = evaluate_direct_dataset(
        dataset_name,
        store,
    )

    (
        k16_metrics,
        k16_predictions,
    ) = evaluate_feature_dataset(
        dataset_name=dataset_name,
        k_unique=16,
        repeat_id=0,
    )

    merged = direct_predictions[
        [
            "sample_key",
            "y_true",
            "p_nonviolence",
            "p_violence",
        ]
    ].merge(
        k16_predictions[
            [
                "sample_key",
                "y_true",
                "p_nonviolence",
                "p_violence",
            ]
        ],
        on="sample_key",
        how="inner",
        suffixes=(
            "_direct",
            "_k16",
        ),
        validate="one_to_one",
    )

    if len(
        merged
    ) != len(
        direct_predictions
    ):
        raise RuntimeError(
            f"{dataset_name}: direct/K16 sample count mismatch."
        )

    if not np.array_equal(
        merged[
            "y_true_direct"
        ].to_numpy(),
        merged[
            "y_true_k16"
        ].to_numpy(),
    ):
        raise RuntimeError(
            f"{dataset_name}: direct/K16 labels mismatch."
        )

    max_prob_delta = float(
        np.max(
            np.abs(
                merged[
                    [
                        "p_nonviolence_direct",
                        "p_violence_direct",
                    ]
                ].to_numpy()
                -
                merged[
                    [
                        "p_nonviolence_k16",
                        "p_violence_k16",
                    ]
                ].to_numpy()
            )
        )
    )

    metric_deltas = {}

    for metric_name in [
        "accuracy",
        "balanced_accuracy",
        "f1_macro",
        "roc_auc",
        "average_precision",
    ]:
        metric_deltas[
            metric_name
        ] = float(
            k16_metrics[
                metric_name
            ]
            -
            direct_metrics[
                metric_name
            ]
        )

    max_metric_delta = float(
        max(
            abs(
                value
            )
            for value
            in metric_deltas.values()
            if np.isfinite(
                value
            )
        )
    )

    gate_rows.append(
        {
            "dataset": dataset_name,
            "max_abs_probability_delta": max_prob_delta,
            "max_abs_metric_delta": max_metric_delta,
            **{
                f"delta_{key}": value
                for key, value
                in metric_deltas.items()
            },
        }
    )

    if (
        max_prob_delta
        > K16_PROBABILITY_ATOL
    ):
        raise RuntimeError(
            f"{dataset_name}: K16 probability reproduction failed: "
            f"{max_prob_delta} > {K16_PROBABILITY_ATOL}"
        )

    if (
        max_metric_delta
        > K16_METRIC_ATOL
    ):
        raise RuntimeError(
            f"{dataset_name}: K16 metric reproduction failed: "
            f"{max_metric_delta} > {K16_METRIC_ATOL}"
        )

    direct_metric_rows.append(
        {
            "seed": RUN_SEED,
            "dataset": dataset_name,
            **direct_metrics,
        }
    )

    direct_prediction_frames.append(
        direct_predictions
    )

    k16_metric_rows.append(
        {
            "seed": RUN_SEED,
            "dataset": dataset_name,
            "k_unique": 16,
            "repeat": 0,
            **k16_metrics,
        }
    )

    k16_prediction_frames.append(
        k16_predictions
    )

    print(
        f"✅ {dataset_name}: "
        f"max Δprob={max_prob_delta:.8g}, "
        f"max Δmetric={max_metric_delta:.8g}"
    )

gate_df = pd.DataFrame(
    gate_rows
)

gate_df.to_csv(
    EVAL_OUTPUT_DIR
    / f"K16_direct_vs_feature_gate_seed{RUN_SEED}.csv",
    index=False,
)

direct_metrics_df = pd.DataFrame(
    direct_metric_rows
)

direct_metrics_df.to_csv(
    EVAL_OUTPUT_DIR
    / f"direct_original_metrics_seed{RUN_SEED}.csv",
    index=False,
)

pd.concat(
    direct_prediction_frames,
    ignore_index=True,
).to_csv(
    EVAL_OUTPUT_DIR
    / f"direct_original_predictions_seed{RUN_SEED}.csv",
    index=False,
)

print(
    "\n✅ Semua K=16 validity gates PASS"
)

display(
    gate_df
)


K=16 VALIDITY GATE | RWF


Direct RWF:   0%|          | 0/500 [00:00<?, ?it/s]

✅ RWF: max Δprob=0, max Δmetric=0

K=16 VALIDITY GATE | UCF


Direct UCF:   0%|          | 0/25 [00:00<?, ?it/s]

✅ UCF: max Δprob=0, max Δmetric=0

K=16 VALIDITY GATE | BV


Direct BV:   0%|          | 0/350 [00:00<?, ?it/s]

✅ BV: max Δprob=0, max Δmetric=0

✅ Semua K=16 validity gates PASS


,dataset,max_abs_probability_delta,max_abs_metric_delta,delta_accuracy,delta_balanced_accuracy,delta_f1_macro,delta_roc_auc,delta_average_precision
0,RWF,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,UCF,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,BV,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 11. Jalankan K=1/2/4/8 × 10 repeats + K=16 reference

In [ ]:
metric_frames = [
    pd.DataFrame(
        k16_metric_rows
    )
]

prediction_frames = [
    pd.concat(
        k16_prediction_frames,
        ignore_index=True,
    )
]

for k_unique in [
    1,
    2,
    4,
    8,
]:
    for repeat_id in range(
        NUM_RANDOM_REPEATS
    ):
        print(
            f"\nseed={RUN_SEED} | "
            f"K={k_unique} | "
            f"repeat={repeat_id}"
        )

        condition_metric_rows = []

        condition_prediction_frames = []

        for dataset_name in HELDOUT_SPECS:
            (
                metrics,
                predictions_df,
            ) = evaluate_feature_dataset(
                dataset_name=dataset_name,
                k_unique=k_unique,
                repeat_id=repeat_id,
            )

            condition_metric_rows.append(
                {
                    "seed": RUN_SEED,
                    "dataset": dataset_name,
                    "k_unique": k_unique,
                    "repeat": repeat_id,
                    **metrics,
                }
            )

            condition_prediction_frames.append(
                predictions_df
            )

        metric_frames.append(
            pd.DataFrame(
                condition_metric_rows
            )
        )

        prediction_frames.append(
            pd.concat(
                condition_prediction_frames,
                ignore_index=True,
            )
        )

all_metrics = pd.concat(
    metric_frames,
    ignore_index=True,
)

all_predictions = pd.concat(
    prediction_frames,
    ignore_index=True,
)

metrics_path = (
    EVAL_OUTPUT_DIR
    / f"frame_diversity_all_repeats_seed{RUN_SEED}.csv"
)

predictions_path = (
    EVAL_OUTPUT_DIR
    / f"frame_diversity_predictions_seed{RUN_SEED}.csv"
)

all_metrics.to_csv(
    metrics_path,
    index=False,
)

all_predictions.to_csv(
    predictions_path,
    index=False,
)

print(
    "\n✅ Semua FrameMean held-out frame-diversity conditions selesai."
)

print(
    "Metrics    :",
    metrics_path,
)

print(
    "Predictions:",
    predictions_path,
)


seed=123 | K=1 | repeat=0

seed=123 | K=1 | repeat=1

seed=123 | K=1 | repeat=2

seed=123 | K=1 | repeat=3

seed=123 | K=1 | repeat=4

seed=123 | K=1 | repeat=5

seed=123 | K=1 | repeat=6

seed=123 | K=1 | repeat=7

seed=123 | K=1 | repeat=8

seed=123 | K=1 | repeat=9

seed=123 | K=2 | repeat=0

seed=123 | K=2 | repeat=1

seed=123 | K=2 | repeat=2

seed=123 | K=2 | repeat=3

seed=123 | K=2 | repeat=4

seed=123 | K=2 | repeat=5

seed=123 | K=2 | repeat=6

seed=123 | K=2 | repeat=7

seed=123 | K=2 | repeat=8

seed=123 | K=2 | repeat=9

seed=123 | K=4 | repeat=0

seed=123 | K=4 | repeat=1

seed=123 | K=4 | repeat=2

seed=123 | K=4 | repeat=3

seed=123 | K=4 | repeat=4

seed=123 | K=4 | repeat=5

seed=123 | K=4 | repeat=6

seed=123 | K=4 | repeat=7

seed=123 | K=4 | repeat=8

seed=123 | K=4 | repeat=9

seed=123 | K=8 | repeat=0

seed=123 | K=8 | repeat=1

seed=123 | K=8 | repeat=2

seed=123 | K=8 | repeat=3

seed=123 | K=8 | repeat=4

seed=123 | K=8 | repeat=5

seed=123 | K=8 | repeat=6



## 12. Summary per K

K=1/2/4/8 diringkas dari 10 temporal-sampling repeats.

K=16 hanya satu deterministic condition, sehingga `std_sampling = 0`.

In [ ]:
METRIC_COLUMNS = [
    "accuracy",
    "balanced_accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "roc_auc",
    "average_precision",
    "precision_nonviolence",
    "recall_nonviolence",
    "f1_nonviolence",
    "precision_violence",
    "recall_violence",
    "f1_violence",
]

summary_rows = []

for (
    dataset_name,
    k_unique,
), group in all_metrics.groupby(
    [
        "dataset",
        "k_unique",
    ]
):
    row = {
        "seed": RUN_SEED,
        "dataset": dataset_name,
        "k_unique": int(
            k_unique
        ),
        "num_sampling_repeats": int(
            len(
                group
            )
        ),
    }

    for metric_name in METRIC_COLUMNS:
        values = group[
            metric_name
        ].astype(
            float
        )

        row[
            f"{metric_name}_mean"
        ] = float(
            values.mean()
        )

        row[
            f"{metric_name}_std_sampling"
        ] = float(
            values.std(
                ddof=1
            )
            if len(
                values
            )
            > 1
            else 0.0
        )

        row[
            f"{metric_name}_min"
        ] = float(
            values.min()
        )

        row[
            f"{metric_name}_max"
        ] = float(
            values.max()
        )

    summary_rows.append(
        row
    )

summary_df = pd.DataFrame(
    summary_rows
).sort_values(
    [
        "dataset",
        "k_unique",
    ]
)

summary_path = (
    EVAL_OUTPUT_DIR
    / f"frame_diversity_summary_seed{RUN_SEED}.csv"
)

summary_df.to_csv(
    summary_path,
    index=False,
)

display(
    summary_df[
        [
            "dataset",
            "k_unique",
            "num_sampling_repeats",
            "average_precision_mean",
            "average_precision_std_sampling",
            "roc_auc_mean",
            "roc_auc_std_sampling",
            "f1_macro_mean",
            "f1_macro_std_sampling",
        ]
    ].round(
        5
    )
)

print(
    "Saved:",
    summary_path,
)

,dataset,k_unique,num_sampling_repeats,average_precision_mean,average_precision_std_sampling,roc_auc_mean,roc_auc_std_sampling,f1_macro_mean,f1_macro_std_sampling
0,BV,1,10,0.49930,0.00925,0.49517,0.01043,0.48225,0.01024
1,BV,2,10,0.50078,0.00672,0.49778,0.00645,0.48046,0.00887
2,BV,4,10,0.50271,0.00323,0.50136,0.00331,0.48338,0.00748
3,BV,8,10,0.50003,0.00262,0.50017,0.00219,0.48025,0.00495
4,BV,16,1,0.50037,0.00000,0.50104,0.00000,0.48273,0.00000
5,RWF,1,10,0.61426,0.01062,0.62277,0.00877,0.56983,0.00661
6,RWF,2,10,0.62635,0.00483,0.63832,0.00445,0.58367,0.00425
7,RWF,4,10,0.63193,0.00610,0.64367,0.00291,0.58845,0.00521
8,RWF,8,10,0.63306,0.00282,0.64664,0.00186,0.59101,0.00259
9,RWF,16,1,0.63076,0.00000,0.64628,0.00000,0.58722,0.00000


Saved: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/FrameMean_FrameDiversity/seed_123/frame_diversity_summary_seed123.csv


## 13. Delta terhadap K=16

Primary paper metric: **Average Precision (AP)**.

Disimpan:
- `ΔAP = AP(K=16) - AP(K)`
- `ΔAUC = AUC(K=16) - AUC(K)`
- `ΔMacroF1 = MacroF1(K=16) - MacroF1(K)`

Nilai positif berarti kondisi K memiliki performa lebih rendah daripada K=16.

In [ ]:
baseline_16 = (
    summary_df[
        summary_df[
            "k_unique"
        ]
        == 16
    ][
        [
            "dataset",
            "average_precision_mean",
            "roc_auc_mean",
            "f1_macro_mean",
        ]
    ]
    .rename(
        columns={
            "average_precision_mean": "ap_k16",
            "roc_auc_mean": "auc_k16",
            "f1_macro_mean": "f1_k16",
        }
    )
)

delta_df = summary_df.merge(
    baseline_16,
    on="dataset",
    how="left",
)

delta_df[
    "delta_ap_k16_minus_k"
] = (
    delta_df[
        "ap_k16"
    ]
    - delta_df[
        "average_precision_mean"
    ]
)

delta_df[
    "delta_auc_k16_minus_k"
] = (
    delta_df[
        "auc_k16"
    ]
    - delta_df[
        "roc_auc_mean"
    ]
)

delta_df[
    "delta_f1_k16_minus_k"
] = (
    delta_df[
        "f1_k16"
    ]
    - delta_df[
        "f1_macro_mean"
    ]
)

delta_df[
    "delta_ap_pp"
] = (
    100.0
    * delta_df[
        "delta_ap_k16_minus_k"
    ]
)

delta_df[
    "delta_auc_pp"
] = (
    100.0
    * delta_df[
        "delta_auc_k16_minus_k"
    ]
)

delta_df[
    "delta_f1_pp"
] = (
    100.0
    * delta_df[
        "delta_f1_k16_minus_k"
    ]
)

delta_path = (
    EVAL_OUTPUT_DIR
    / f"frame_diversity_delta_seed{RUN_SEED}.csv"
)

delta_df.to_csv(
    delta_path,
    index=False,
)

display(
    delta_df[
        [
            "dataset",
            "k_unique",
            "average_precision_mean",
            "delta_ap_pp",
            "roc_auc_mean",
            "delta_auc_pp",
            "f1_macro_mean",
            "delta_f1_pp",
        ]
    ].round(
        4
    )
)

print(
    "Saved:",
    delta_path,
)

,dataset,k_unique,average_precision_mean,delta_ap_pp,roc_auc_mean,delta_auc_pp,f1_macro_mean,delta_f1_pp
0,BV,1,0.4993,0.1069,0.4952,0.5864,0.4822,0.0480
1,BV,2,0.5008,-0.0409,0.4978,0.3256,0.4805,0.2266
2,BV,4,0.5027,-0.2338,0.5014,-0.0320,0.4834,-0.0652
3,BV,8,0.5000,0.0346,0.5002,0.0865,0.4803,0.2477
4,BV,16,0.5004,0.0000,0.5010,0.0000,0.4827,0.0000
5,RWF,1,0.6143,1.6498,0.6228,2.3508,0.5698,1.7388
6,RWF,2,0.6263,0.4407,0.6383,0.7962,0.5837,0.3547
7,RWF,4,0.6319,-0.1173,0.6437,0.2615,0.5885,-0.1234
8,RWF,8,0.6331,-0.2302,0.6466,-0.0357,0.5910,-0.3789
9,RWF,16,0.6308,0.0000,0.6463,0.0000,0.5872,0.0000


Saved: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/FrameMean_FrameDiversity/seed_123/frame_diversity_delta_seed123.csv


## 14. Protocol manifest / audit

In [ ]:
protocol_audit = {
    "task": "heldout_frame_diversity_evaluation_only",
    "variant": VARIANT,
    "checkpoint_seed": RUN_SEED,
    "checkpoint_path": str(
        CHECKPOINT_PATH
    ),
    "checkpoint_epoch": checkpoint.get(
        "epoch"
    ),
    "training_performed": False,
    "fine_tuning_performed": False,
    "strict_checkpoint_loading": True,
    "heldout_datasets": list(
        HELDOUT_SPECS.keys()
    ),
    "sequence_length_cache": SEQUENCE_LENGTH,
    "unique_frame_counts": K_VALUES,
    "sampling_repeats_for_k_lt_16": NUM_RANDOM_REPEATS,
    "sampling_repeats_for_k16": 1,
    "sampling_method": (
        "divide 16 temporal positions into K equal bins; "
        "sample one position uniformly from each bin"
    ),
    "sampling_base_seed": SAMPLING_BASE_SEED,
    "sampling_depends_on_checkpoint_seed": False,
    "sampling_identity_inputs": [
        "sampling_base_seed",
        "repeat_id",
        "dataset_name_crc32",
        "sample_index",
        "k_unique",
    ],
    "frame_mean_backend": "PyTorch mean on device",
    "classifier_backend": "PyTorch on device",
    "feature_cache_dtype": "float32",
    "reference_batch_size": REFERENCE_BATCH_SIZE,
    "use_amp": USE_AMP,
    "k16_direct_vs_feature_gate": True,
    "k16_probability_atol": K16_PROBABILITY_ATOL,
    "k16_metric_atol": K16_METRIC_ATOL,
    "important_note_k1": (
        "K=1 is not the separately trained SingleFrame model."
    ),
    "equal_duplication_equivalence": (
        "For FrameMean and K dividing 16, mean over K selected features "
        "equals mean after equal duplication of each selected feature to T=16."
    ),
    "primary_metric": "average_precision",
    "secondary_metrics": [
        "roc_auc",
        "f1_macro",
    ],
    "no_cross_dataset_pooling": True,
}

manifest_path = (
    EVAL_OUTPUT_DIR
    / f"protocol_audit_seed{RUN_SEED}.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        protocol_audit,
        file,
        indent=2,
    )

print(
    json.dumps(
        protocol_audit,
        indent=2,
    )
)

print(
    "\nSemua output tersimpan di:",
    EVAL_OUTPUT_DIR,
)

{
  "task": "heldout_frame_diversity_evaluation_only",
  "variant": "frame_mean",
  "checkpoint_seed": 123,
  "checkpoint_path": "/content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/EffTempNet_FrameMean_NoLS_123Seeds_v1/frame_mean/seed_123/best.pt",
  "checkpoint_epoch": 17,
  "training_performed": false,
  "fine_tuning_performed": false,
  "strict_checkpoint_loading": true,
  "heldout_datasets": [
    "RWF",
    "UCF",
    "BV"
  ],
  "sequence_length_cache": 16,
  "unique_frame_counts": [
    1,
    2,
    4,
    8,
    16
  ],
  "sampling_repeats_for_k_lt_16": 10,
  "sampling_repeats_for_k16": 1,
  "sampling_method": "divide 16 temporal positions into K equal bins; sample one position uniformly from each bin",
  "sampling_base_seed": 20260810,
  "sampling_depends_on_checkpoint_seed": false,
  "sampling_identity_inputs": [
    "sampling_base_seed",
    "repeat_id",
    "dataset_name_crc32",
    "sample_index",
    "k_unique"
  ],
  "frame_mean_backend": "PyTorch mean on de

# Seed berikutnya

Setelah seed pertama selesai, **cukup ubah**:

```python
RUN_SEED = 0
```

menjadi:

```python
RUN_SEED = 42
```

lalu:

```python
RUN_SEED = 123
```

Jangan ubah:
```python
K_VALUES = [1, 2, 4, 8, 16]
NUM_RANDOM_REPEATS = 10
SAMPLING_BASE_SEED = 20260810
SEQUENCE_LENGTH = 16
REFERENCE_BATCH_SIZE = 4
```

Raw held-out cache tetap shared.

Feature cache otomatis disimpan per checkpoint seed:

```text
HeldOut_Evaluation/
└── FrameMean_FrameDiversity/
    ├── seed_0/
    │   └── frame_feature_cache/
    ├── seed_42/
    │   └── frame_feature_cache/
    └── seed_123/
        └── frame_feature_cache/
```

Output utama per seed:
- `frame_diversity_all_repeats_seedX.csv`
- `frame_diversity_predictions_seedX.csv`
- `frame_diversity_summary_seedX.csv`
- `frame_diversity_delta_seedX.csv`
- `K16_direct_vs_feature_gate_seedX.csv`
- `heldout_dataset_audit.csv`
- `protocol_audit_seedX.json`

`frame_diversity_predictions_seedX.csv` sudah menyimpan:
- `sample_key`
- `K`
- `repeat`
- selected frame indices
- `y_true`
- `y_pred`
- `is_correct`
- `p_violence`

sehingga dapat dipakai untuk paired error analysis / McNemar pada kondisi yang
benar-benar matching.